In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
# uv add langchain-community pdfplumber
from langchain_community.document_loaders import PDFPlumberLoader

loader = PDFPlumberLoader("../data/Employee_Benefits_Guide_2026_v1.pdf")
documents = loader.load()

In [7]:
documents[0].page_content

'2026 년 테크노빌드 주식회사(TechnoBuild) 임직원\n통합 가이드북\n문서 번호: TB-HR-2026-001 (Rev 5.0)\n발행일: 2026 년 1 월 15 일\n적용 대상: 테크노빌드 전 임직원 (정규직, 계약직, 파견직 포함)\n주관 부서: 인사문화본부 (HR Culture Division)\n보안 등급: 대외비 (Internal Use Only)\n[목 차]\n1. 회사 개요 (Company Overview)\no 1.1 CEO 인사말\no 1.2 기업 미션 및 비전\no 1.3 핵심 가치 : T-SPIRIT\no 1.4 조직도 및 본부 소개\n2. 인사 및 평가 제도 (HR System)\no 2.1 직급 및 호칭 체계\no 2.2 승진 포인트 제도 (Tech-Point)\no 2.3 성과 평가 (T-MBO) 프로세스\no 2.4 전문직군 전환 제도 (Dual Ladder)\n3. 근무 및 복무 관리 (Work & Duty)\no 3.1 표준 근무 시간 및 유연 근무제\no 3.2 생산직 교대 근무 (T-Shift) 운영 규정\no 3.3 시간외 근무 및 특근 신청 절차\no 3.4 휴가 제도 상세 (연차, 공가, 병가, 청원 휴가 등)\no 3.5 하이브리드 및 재택 근무 가이드 (Hybrid Work Policy)\n'

In [5]:
# uv add langchain-text-splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=100)
texts = text_splitter.split_documents(documents)

In [8]:
texts[0].page_content

'2026 년 테크노빌드 주식회사(TechnoBuild) 임직원\n통합 가이드북\n문서 번호: TB-HR-2026-001 (Rev 5.0)\n발행일: 2026 년 1 월 15 일\n적용 대상: 테크노빌드 전 임직원 (정규직, 계약직, 파견직 포함)\n주관 부서: 인사문화본부 (HR Culture Division)\n보안 등급: 대외비 (Internal Use Only)\n[목 차]\n1. 회사 개요 (Company Overview)\no 1.1 CEO 인사말\no 1.2 기업 미션 및 비전\no 1.3 핵심 가치 : T-SPIRIT\no 1.4 조직도 및 본부 소개\n2. 인사 및 평가 제도 (HR System)\no 2.1 직급 및 호칭 체계\no 2.2 승진 포인트 제도 (Tech-Point)'

In [9]:
texts[1].page_content

'o 1.4 조직도 및 본부 소개\n2. 인사 및 평가 제도 (HR System)\no 2.1 직급 및 호칭 체계\no 2.2 승진 포인트 제도 (Tech-Point)\no 2.3 성과 평가 (T-MBO) 프로세스\no 2.4 전문직군 전환 제도 (Dual Ladder)\n3. 근무 및 복무 관리 (Work & Duty)\no 3.1 표준 근무 시간 및 유연 근무제\no 3.2 생산직 교대 근무 (T-Shift) 운영 규정\no 3.3 시간외 근무 및 특근 신청 절차\no 3.4 휴가 제도 상세 (연차, 공가, 병가, 청원 휴가 등)\no 3.5 하이브리드 및 재택 근무 가이드 (Hybrid Work Policy)'

In [10]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_classic.storage import LocalFileStore

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

store = LocalFileStore("./cache/")

cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    embeddings,
    store,
    namespace=embeddings.model
)

d:\wsh\langchain-workspace\ch06\.venv\Lib\site-packages\langchain_classic\embeddings\cache.py:58: UserWarning: Using default key encoder: SHA-1 is *not* collision-resistant. While acceptable for most cache scenarios, a motivated attacker can craft two different payloads that map to the same cache key. If that risk matters in your environment, supply a stronger encoder (e.g. SHA-256 or BLAKE2) via the `key_encoder` argument. If you change the key encoder, consider also creating a new cache, to avoid (the potential for) collisions with existing keys.
  _warn_about_sha1_encoder()


In [11]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(texts, cached_embedder) # in-memory

In [15]:
retriver = vector_store.as_retriever()

In [21]:
docs = retriver.invoke("결혼 하게되면 얼마 받아?")

In [24]:
print("\n\n".join([doc.page_content for doc in docs]))

가족 수당 부양 가족이 있는 자 배우자 5 만원, 등본 제출 필수
자녀당 3 만원 / 월
4.4 특수 영입 및 장기 근속 보너스 (Signing Bonus)
테크노빌드는 핵심 기술 인력 확보를 위해 입사 시점에 일시불로 지급되는 사이닝 보너스
제도를 운영합니다.
A. 지급 대상 및 금액
1. 대상: 연구직 G3 이상 또는 사외 특급 기술 자격 보유자 중 채용위원회 승인을 득한
자.
2. 지급액: 전 직장 연봉의 10% ~ 50% 범위 내에서 개별 협의.
3. 지급 시기: 입사 첫 달 급여일에 일괄 지급.
B. 의무 근속 및 환수(Clawback) 조항
지급 후 36 개월(3 년) 이내에 개인 사유로 퇴사할 경우, 미충족 기간에 대해 월할 계산하여
반환해야 합니다.
(36−실제근속개월수)

없는 지역으로 외근 시에는 ‘법인 카드’ 또는 ’교통비 영수증 청구’가 가능합니다. (팀장
전결 필요)
Q: 사택(기숙사) 입주 조건은 어떻게 되나요? A: 원거리 거주자(회사 반경 40km 이외 지역
거주) 및 독신자 우선 배정입니다. 기혼자도 ‘주말 부부’ 등의 사유가 인정되면 입주
가능합니다. 월 관리비 5 만원은 급여에서 공제되며, 최대 거주 기간은 2 년(1 회 연장 가능,
최대 4 년)입니다.

한도 내 실비 지원 (환율은 신청일 매매기준율 적용).
5.2 주거 안정 자금 대출 (Sweet Home Loan)
 용도: 주택 구입(본인 명의), 전세/월세 보증금.
 한도:
o 근속 1~3 년: 3,000 만원
o 근속 3~5 년: 5,000 만원
o 근속 5 년 이상: 최대 1 억원
 금리: 연 1.5% 고정 금리 (회사 복지 기금에서 이차 보전).

 소멸: 당해 연도 12 월 31 일까지 미사용 시 자동 소멸 (이월 불가).
5.4 경조사 지원 기준 (Family Support Details)
기쁨과 슬픔을 함께 나누는 테크노빌드의 경조 지원입니다.
구분 대상 휴가 (일수) 경조금 (만원) 화환/조화
결혼 본인 5 일 100 + 화환 지원
자녀 1 일 50

In [18]:
vector_store.similarity_search("부모님이 돌아가셨는데 휴가는 몇 일이야?", k=1)

[Document(id='ae0fca9d-7cff-46ed-a8d4-f01e14c14bba', metadata={'source': '../data/Employee_Benefits_Guide_2026_v1.pdf', 'file_path': '../data/Employee_Benefits_Guide_2026_v1.pdf', 'page': 11, 'total_pages': 22, 'Author': '', 'CreationDate': "D:20260116002528+09'00'", 'ModDate': "D:20260116002528+09'00'", 'Producer': 'Microsoft: Print To PDF', 'Title': 'Microsoft Word - Employee_Benefits_Guide_2026_v1.docx'}, page_content='\uf0b7 소멸: 당해 연도 12 월 31 일까지 미사용 시 자동 소멸 (이월 불가).\n5.4 경조사 지원 기준 (Family Support Details)\n기쁨과 슬픔을 함께 나누는 테크노빌드의 경조 지원입니다.\n구분 대상 휴가 (일수) 경조금 (만원) 화환/조화\n결혼 본인 5 일 100 + 화환 지원\n자녀 1 일 50 + 화환 지원\n30 -\n형제/자매 1 일\n30 -\n회갑/칠순 본인/배우자 부모 1 일\n출산 본인 출산휴가 (90 일) 출산 축하금 50 과일 바구니\n배우자 10 일 (유급) 출산 축하금 50 과일 바구니\n-\n사망 본인/배우자 500 + 장례용품 3 단 조화 + 근조기\n부모/배우자 부모 5 일 100 + 장례용품 3 단 조화 + 근조기\n30\n조부모/외조부모 3 일 조화\n30\n형제/자매 3 일 조화')]

In [25]:
from langchain.tools import tool

retriver = vector_store.as_retriever()

@tool
def search_documents(query: str) -> str:
    """테크노빌드 주식회사 임직원 통합 가이드북 정보를 검색합니다."""
    docs = retriver.invoke(query)

    return "\n\n".join([doc.page_content for doc in docs])

In [50]:
system_prompt="""당신은 임직원 통합 가이드북 정보를 제공하는 인사 담당자입니다.

1. 정보가 필요할 경우 반드시 검색 도구를 사용하여 확인하세요.
2. 답변은 반드시 검색된 문서의 내용에만 기반하여 작성하세요.
3. 문서에 관련 내용이 없다면 억지로 꾸며내지 말고 모른다고 답변하세요.
"""

In [51]:
system_prompt="""당신은 임직원 통합 가이드북 정보를 제공하는 인사 담당자입니다.

1. 정보가 필요할 경우 검색 도구를 사용하여 확인하세요.
2. 일반적이거나 이미 학습된 내용은 스스로 답변을 생성하세요.
3. 임직원 통합 가이드북 정보의 검색이 필요한 경우에는 도구를 사용하세요.
"""

In [52]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    tools=[search_documents],
    system_prompt=system_prompt
)

In [29]:
query = "부모님이 돌아가셨는데 휴가는 몇 일이야?"

In [55]:
res = agent.invoke({"messages": [{"role": "user", "content": query}]})

In [56]:
res

{'messages': [HumanMessage(content='부모님이 돌아가셨는데 휴가는 몇 일이야?', additional_kwargs={}, response_metadata={}, id='e6628af1-3133-44e7-afa6-2d61ceeb341a'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'search_documents', 'arguments': '{"query": "\\ubd80\\ubaa8\\ub2d8 \\uc0ac\\ub9dd \\uc2dc \\ud734\\uac00"}'}, '__gemini_function_call_thought_signatures__': {'005640f8-49f8-4e62-a987-ad2c568bdd19': 'CugDAXLI2nyo8qHFfz10O/MM78IumuoPqUqAUofUCY5ikso16uk3PuYic5qbAVESUB1aNcBHm02GyZ0R+Ei5l7RQxvGyLAs0jAViCm31vBfUILZXw2WMsEmupY5fF4haf0yd9AI7H11FAM0H8LPUG/044DHNFE5iCMFlvNBJuoBYbia0yPkQ4jzlILqx4UqkJfe1koaQCoQPszr61mK1WQ9HceP/Xy0IL6aD8rSk+cXt2uWA06GoyaugfJtu2fRw+b5YF4eQd0rsjsuiytsUXI63CROzOm74Vt+9wagkzzGYWhREFjfDy4n555mwJ4pD1ne/HklmhcIKBkGKDhMRbw8Wms/nbJjcCYX4ZvPggjR92fY6YwRP7lrS8+axUaC5Ff6seiPl2JarHnINP/CMe0VvM0eOgyL0qalbehBjElt5ZSIC33LihMAJjj89Eue4MjmqpXIyA5fxYEgoxm+jfqTqrTmRsJGBS74LLrWHVU4Urmk+N9U0nG2Acyrv8Rcyrtp9rrejXCG1yGwjXoac+KZ7lsfTXoSfaMc0qM1Bhvy6E20pZUbkPsz/N4ZVUWCL7KRcqQ8

In [34]:
res['messages'][-1]

AIMessage(content=[{'type': 'text', 'text': '부모님이 돌아가셨을 경우 5일의 휴가가 제공됩니다.', 'extras': {'signature': 'Cu4CAXLI2nwqA/B/qKZiw6qrUGC3pvfWnAODr1mGEMBbAuZ9vj7IdBKGZ32xVrvX4XuFB4i4aKP1F6JhH+qaJOeMoQRCIixNjvuAOOdY+kJd0uXTl/sI7E2QzBeBPfafP7z45EA/QOaBw0Sec2TESa00pwJA8rsUF6HfnR5xCabP7t1BaIMdb2W7D+i2Aip+Y9AYl4R/t1K/iFMcyCegxnwMsRGCXsUIiPaFo53Wrjbs4zwtbU06gUKAef9EM3hWV8OOPRcIAgef89OwolKSM4XWbEbGieV6/M7as1pPLyUDhvHXMturhjLKAUb5cye/P/uWthysGiFMMdhBjXqsVgohMg/GKO+9x1akEDfyEZv6zSYvHcsC6kVdrzDTY/Z54vHSnROeqLgmJCCO++AFFhEMTDMDI+vt9HlW8XmNk4MOiZFxayCwePjHKFb7w9q0js6FyhsiB1PmlB4Bk24mK3ZUDcSjaNoxlxjmRleDscGR'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019beea2-88ea-7120-a5b7-d8e2fd922b0b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1033, 'output_tokens': 106, 'total_tokens': 1139, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'rea

In [53]:
res = agent.invoke({"messages": [{"role": "user", "content": "랭체인 대해 설명해줘."}]})

In [54]:
res

{'messages': [HumanMessage(content='랭체인 대해 설명해줘.', additional_kwargs={}, response_metadata={}, id='45a170f2-6a2d-4bd3-b30f-f927c5ce9555'),
  AIMessage(content=[{'type': 'text', 'text': '랭체인은 대규모 언어 모델(LLM)을 사용하여 애플리케이션을 개발하는 것을 돕는 프레임워크입니다. LLM을 다양한 외부 데이터 소스와 연결하고, 여러 구성 요소를 조합하여 복잡한 애플리케이션을 만들 수 있도록 설계되었습니다.\n\n주요 특징은 다음과 같습니다:\n\n*   **컴포넌트:** LLM과 상호작용하는 데 필요한 다양한 모듈(모델, 프롬프트, 파서 등)을 제공합니다.\n*   **체인:** 이러한 컴포넌트들을 연결하여 특정 작업을 수행하는 일련의 단계를 구성할 수 있습니다. 예를 들어, 사용자 질문을 받아 문서를 검색하고, 그 내용을 바탕으로 LLM이 답변을 생성하도록 할 수 있습니다.\n*   **에이전트:** LLM이 어떤 도구를 사용해야 할지 스스로 결정하고, 주어진 목표를 달성하기 위해 일련의 행동을 계획하고 실행할 수 있도록 돕습니다.\n*   **메모리:** 이전 대화 내용을 기억하여 더 자연스럽고 연속적인 상호작용이 가능하게 합니다.\n\n랭체인을 사용하면 질의응답 시스템, 챗봇, 요약 도구 등 다양한 LLM 기반 애플리케이션을 효율적으로 구축할 수 있습니다.', 'extras': {'signature': 'CsYCAXLI2nzAVyZZ+cLn/7BbUXVEZzwFkpA2zlqvrPRRG3/6NRKYEIc1S3ntpCnTgBGUFsxFqtJn5SHTLvEII6uP51TR70l99CkTOXQEvtzr2s/WRnPxt2NKU8rMjeow1I6YvcVajPjq/YG9tLFgL0VR9w8/3oW7K13y8Fc9o9xvAF13qVvsWj6/F2wSPbjW4BBhkXjGfl3+SnHdc1fu/CqpV/AfpB9FZauhjS